[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/allanspadini/curso-vision-transformers-infnet/blob/main/aula_08_avaliacao_fid_e_sintese_visao_computacional/aula_08_metricas_fid_e_sintese_visao_computacional.ipynb)

# 🎓 Aula 08: Avaliação Quantitativa de GANs (FID) e Síntese Teórica da Visão Computacional
### **Faculdade Infnet — Pós-Graduação em Visão Computacional com CNNs e Transformers**
**Professor:** Dr. Allan Spadini  
**Metodologia Pedagógica Central:** Situação-Problema do Mundo Real ➔ Solução de Engenharia ➔ Teoria Rigorosa

---

### 🎯 Objetivos Deste Laboratório:
1. **Métrica FID (Fréchet Inception Distance) do Zero em PyTorch:** Implementar a extração de ativações na camada `pool3` ($d=2048$) da Inception-v3 pré-treinada e o cálculo da distância Wasserstein-2 entre distribuições multivariadas reais e sintéticas.
2. **Interpretabilidade de Vision Transformers (ViT):** Construir o pipeline de hooks em PyTorch para extrair a matriz de atenção da camada final, isolar a ponderação do token `[CLS]`, remodelar os patches espaciais ($14 \times 14$) e sobrepor o mapa de calor à imagem original.
3. **Busca Semântica Multimodal com CLIP:** Manipular embeddings normalizados $L_2$ de imagens e textos, calcular similaridades angulares de cosseno e calibrar empiricamente limiares de corte (thresholds $\tau$) para detecção conceitual.
4. **Auditoria de Desbalanceamento Severo & Governança Downstream:** Simular o Paradoxo da Acurácia sob escassez de dados, avaliar matrizes de confusão, calcular o salto em Recall (Sensibilidade) da classe minoritária e aplicar o protocolo científico de validação em teste 100% real.
5. **Boas Práticas Científicas:** Eliminação de vazamento de dados por amostragem correlacionada (*Group Split*) e estratégias contra *Covariate Shift*.


---
## ⚙️ Setup do Ambiente, GPU CUDA e Reprodutibilidade

Verificamos a disponibilidade de aceleração por GPU (Google Colab T4) e fixamos sementes para garantia de reprodutibilidade científica.


In [ ]:
# Instalação de dependências essenciais
!pip install -q torchvision scikit-learn scipy matplotlib seaborn tqdm open-clip-torch


In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import linalg
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from torchvision.models import Inception_V3_Weights, ViT_B_16_Weights

# Fixação estrita de sementes para reprodutibilidade
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Dispositivo de execução: {device}")
if device.type == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM Disponível: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


---
## 🔬 Módulo 1: Implementação Numérica da Métrica FID (Fréchet Inception Distance)

### 1.1 O Formalismo Matemático
O Fréchet Inception Distance mede a distância de Wasserstein-2 entre duas distribuições normais multivariadas $\mathcal{N}(\mu_r, \Sigma_r)$ (dados reais) e $\mathcal{N}(\mu_g, \Sigma_g)$ (dados gerados):

$$\text{FID} = \|\mu_r - \mu_g\|_2^2 + \text{Tr}\left(\Sigma_r + \Sigma_g - 2(\Sigma_r \Sigma_g)^{1/2}\right)$$

Onde as features são extraídas na camada de global average pooling (`pool3`, $d=2048$) da rede Inception-v3.


In [ ]:
def calculate_frechet_distance(mu1, sigma1, mu2, sigma2, eps=1e-6):
    """
    Calcula a distância de Fréchet entre duas gaussianas multivariadas N(mu1, sigma1) e N(mu2, sigma2).
    """
    mu1 = np.atleast_1d(mu1)
    mu2 = np.atleast_1d(mu2)
    sigma1 = np.atleast_2d(sigma1)
    sigma2 = np.atleast_2d(sigma2)

    assert mu1.shape == mu2.shape, f"Dimensões de média incompatíveis: {mu1.shape} vs {mu2.shape}"
    assert sigma1.shape == sigma2.shape, f"Dimensões de covariância incompatíveis: {sigma1.shape} vs {sigma2.shape}"

    # 1. Termo de deslocamento euclidiano de centroide (Fidelidade média)
    diff = mu1 - mu2
    mean_term = np.dot(diff, diff)

    # 2. Produto de matrizes de covariância
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)

    # Tratamento numérico para singularidades e resíduos imaginários
    if not np.isfinite(covmean).all():
        offset = np.eye(sigma1.shape[0]) * eps
        covmean = linalg.sqrtm((sigma1 + offset).dot(sigma2 + offset))

    if np.iscomplexobj(covmean):
        if not np.allclose(np.diagonal(covmean).imag, 0, atol=1e-3):
            max_imag = np.max(np.abs(covmean.imag))
            print(f"⚠️ Aviso: Componente imaginário residual relevante detectado: {max_imag:.4e}")
        covmean = covmean.real

    # 3. Termo de divergência de covariância (Diversidade & Correlação)
    cov_term = np.trace(sigma1 + sigma2 - 2.0 * covmean)

    fid = mean_term + cov_term
    return float(fid), float(mean_term), float(cov_term)

print("✅ Função calculate_frechet_distance implementada com sucesso!")


### 1.2 Extração de Features Inception-v3 (`pool3`, $d=2048$)

Construímos um extrator de características congelado a partir do Inception-v3 pré-treinado.


In [ ]:
class InceptionFeatureExtractor(nn.Module):
    """
    Extrator de representações latentes contínuas da penúltima camada (pool3, 2048-D) do Inception-v3.
    """
    def __init__(self):
        super().__init__()
        weights = Inception_V3_Weights.DEFAULT
        self.inception = models.inception_v3(weights=weights)
        self.inception.fc = nn.Identity()  # Substitui o classificador linear por identidade
        self.inception.eval()
        for param in self.inception.parameters():
            param.requires_grad = False

        # Transformação oficial de pré-processamento ImageNet (299x299)
        self.transforms = weights.transforms()

    def forward(self, x):
        # x: [B, 3, 299, 299] normalizado
        return self.inception(x)  # Retorna [B, 2048]

inception_extractor = InceptionFeatureExtractor().to(device)
print("✅ Extrator Inception-v3 (pool3, 2048-D) pronto para inferência.")


### 1.3 Simulação Experimental: Avaliação de Distribuições Idênticas vs Mode Collapse vs Ruído

Vamos testar a sensibilidade da métrica FID gerando amostras de teste e perturbando sua média e dispersão.


In [ ]:
# Criamos um conjunto de 500 imagens simuladas de dimensão 299x299
N_SAMPLES = 400
dummy_real_images = torch.clamp(torch.randn(N_SAMPLES, 3, 299, 299) * 0.2 + 0.5, 0.0, 1.0).to(device)

# Extração de features do conjunto real
with torch.no_grad():
    feat_real = inception_extractor(dummy_real_images).cpu().numpy()

mu_real = np.mean(feat_real, axis=0)
sigma_real = np.cov(feat_real, rowvar=False)

# Cenário 1: Distribuição Idêntica (Amostras do mesmo conjunto)
feat_ident = feat_real[:N_SAMPLES // 2]
mu_ident = np.mean(feat_ident, axis=0)
sigma_ident = np.cov(feat_ident, rowvar=False)
fid_ident, m_ident, c_ident = calculate_frechet_distance(mu_real, sigma_real, mu_ident, sigma_ident)

# Cenário 2: Deslocamento de Média (Desvio de cor/iluminação)
dummy_shifted = torch.clamp(dummy_real_images + 0.25, 0.0, 1.0)
with torch.no_grad():
    feat_shift = inception_extractor(dummy_shifted).cpu().numpy()
mu_shift = np.mean(feat_shift, axis=0)
sigma_shift = np.cov(feat_shift, rowvar=False)
fid_shift, m_shift, c_shift = calculate_frechet_distance(mu_real, sigma_real, mu_shift, sigma_shift)

# Cenário 3: Mode Collapse (Apenas 5 imagens repetidas múltiplas vezes)
dummy_collapsed = dummy_real_images[:5].repeat(80, 1, 1, 1)
with torch.no_grad():
    feat_collapse = inception_extractor(dummy_collapsed).cpu().numpy()
mu_collapse = np.mean(feat_collapse, axis=0)
sigma_collapse = np.cov(feat_collapse, rowvar=False)
fid_collapse, m_collapse, c_collapse = calculate_frechet_distance(mu_real, sigma_real, mu_collapse, sigma_collapse)

print(f"📊 Resultados Comparativos do FID:")
print(f"  1. Distribuição Idêntica:   FID = {fid_ident:8.2f} (Termo Média: {m_ident:6.2f} | Termo Covariância: {c_ident:6.2f})")
print(f"  2. Deslocamento de Média:   FID = {fid_shift:8.2f} (Termo Média: {m_shift:6.2f} | Termo Covariância: {c_shift:6.2f})")
print(f"  3. Mode Collapse Severo:    FID = {fid_collapse:8.2f} (Termo Média: {m_collapse:6.2f} | Termo Covariância: {c_collapse:6.2f})")


---
## 👁️ Módulo 2: Vision Transformers (ViT) — Extração de Mapas de Atenção e Interpretabilidade

### 2.1 Mecânica do Hook em PyTorch
Em um ViT-B/16 com imagens de $224 \times 224$, temos $N = (224/16)^2 = 196$ patches espaciais + 1 token `[CLS]` (total 197 tokens).
A matriz de atenção da última camada tem dimensões $[B, h, 197, 197]$.
A relevância que a classificação final atribui a cada patch é extraída da **linha 0** (token `[CLS]`): $A[:, h, 0, 1:] \in \mathbb{R}^{196}$.


In [ ]:
# Carregamos o ViT-B/16 pré-treinado no ImageNet
vit_weights = ViT_B_16_Weights.DEFAULT
vit_model = models.vit_b_16(weights=vit_weights).to(device)
vit_model.eval()

# Dicionário para armazenar a matriz de atenção capturada pelo hook
attention_store = {}

def attention_hook(module, input, output):
    # No PyTorch MultiheadAttention, output[1] contém a matriz de atenção média ou por cabeça
    # Interceptamos os pesos de atenção calculados no forward
    pass

print("✅ ViT-B/16 carregado com sucesso!")


In [ ]:
# Implementação robusta de extração direta de atenção via forward manual do bloco final
def extract_vit_attention(model, x):
    """
    Executa o forward pass do ViT e extrai os attention weights da última camada para o token [CLS].
    """
    # x: [1, 3, 224, 224]
    with torch.no_grad():
        # 1. Patch projection e token prepending
        x_proc = model._process_input(x)  # [1, 196, 768]
        n = x_proc.shape[0]
        batch_class_token = model.class_token.expand(n, -1, -1)
        x_tokens = torch.cat([batch_class_token, x_proc], dim=1)
        x_tokens = x_tokens + model.encoder.pos_embedding

        # 2. Passagem até a penúltima camada do encoder
        for layer in model.encoder.layers[:-1]:
            x_tokens = layer(x_tokens)

        # 3. Na última camada, interceptamos a MultiheadAttention
        last_layer = model.encoder.layers[-1]
        norm_tokens = last_layer.ln_1(x_tokens)

        # Executamos o MHA retornando attention weights explicitamente
        attn_output, attn_weights = last_layer.self_attention(
            norm_tokens, norm_tokens, norm_tokens, need_weights=True, average_attn_weights=False
        )
        # attn_weights: [B, num_heads, 197, 197]
        return attn_weights

print("✅ Função extract_vit_attention definida com sucesso!")


In [ ]:
# Criamos uma imagem sintética estruturada com um objeto circular de alto contraste no centro
test_img = np.ones((224, 224, 3), dtype=np.float32) * 0.2
rr, cc = np.ogrid[:224, :224]
mask = (rr - 112) ** 2 + (cc - 112) ** 2 <= 40 ** 2
test_img[mask] = [0.9, 0.2, 0.2]  # Círculo vermelho central

# Adicionamos ruído leve de fundo
test_img += np.random.normal(0, 0.03, test_img.shape)
test_img = np.clip(test_img, 0.0, 1.0)

# Conversão para tensor normalizado ImageNet
norm_fn = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
tensor_img = norm_fn(torch.from_numpy(test_img).permute(2, 0, 1).float()).unsqueeze(0).to(device)

# Extração da matriz de atenção
attn_matrix = extract_vit_attention(vit_model, tensor_img)
# attn_matrix: [1, 12, 197, 197]

# Extraímos a média de todas as 12 cabeças para a linha do token [CLS] (índice 0) nos 196 patches (1:197)
cls_attn_heads = attn_matrix[0, :, 0, 1:].cpu().numpy()  # [12, 196]
cls_attn_mean = np.mean(cls_attn_heads, axis=0)          # [196]

# Remodelagem espacial: [196] -> [14, 14]
heat_grid = cls_attn_mean.reshape(14, 14)

# Interpolação bilinear: [14, 14] -> [224, 224]
heat_tensor = torch.from_numpy(heat_grid).unsqueeze(0).unsqueeze(0).float()
heat_upsampled = F.interpolate(heat_tensor, size=(224, 224), mode='bilinear', align_corners=False).squeeze().numpy()

# Visualização lado a lado: Imagem Original, Heatmap e Sobreposição
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(test_img)
axes[0].set_title("1. Imagem de Entrada Original", fontsize=12, fontweight='bold')
axes[0].axis('off')

im1 = axes[1].imshow(heat_upsampled, cmap='jet')
axes[1].set_title("2. Attention Heatmap ([CLS] Token Line 0)", fontsize=12, fontweight='bold')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
axes[1].axis('off')

axes[2].imshow(test_img)
axes[2].imshow(heat_upsampled, cmap='jet', alpha=0.55)
axes[2].set_title("3. Sobreposição de Interpretabilidade", fontsize=12, fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
plt.show()
print("✅ Mapa de atenção extraído e renderizado com sucesso!")


---
## 🌐 Módulo 3: Busca Semântica Multimodal com CLIP e Calibração de Thresholds

### 3.1 Projeção na Hiper-Esfera Unitária e Similaridade de Cosseno
No CLIP, tanto as features de imagem $v$ quanto as features de texto $u$ são normalizadas pela norma $L_2$:

$$\tilde{v} = \frac{v}{\|v\|_2}, \quad \tilde{u} = \frac{u}{\|u\|_2} \implies S = \tilde{v}^T \tilde{u} = \cos(\theta)$$


In [ ]:
import open_clip

# Carregamos o modelo CLIP pré-treinado oficial (ViT-B-32)
clip_model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
clip_model = clip_model.to(device)
clip_model.eval()
tokenizer = open_clip.get_tokenizer('ViT-B-32')

print("✅ CLIP ViT-B-32 carregado na GPU/CPU com sucesso!")


In [ ]:
# Lista de consultas textuais variando em especificidade e nível de abstração
queries = [
    "a photo of a red sports car on a highway",
    "a microscopic view of biological cells",
    "a crowded urban street with vehicles",
    "a feeling of freedom, open sky and solitude",
    "healthy organic vegetables and diet food"
]

# Tokenização e extração de embeddings textuais normalizados
text_tokens = tokenizer(queries).to(device)
with torch.no_grad():
    text_features = clip_model.encode_text(text_tokens)
    # Normalização L2 obrigatória
    text_features = F.normalize(text_features, p=2, dim=-1)

print(f"📐 Formato dos embeddings textuais normalizados L2: {text_features.shape}")
print(f"   Norma euclidiana do vetor 0: {torch.norm(text_features[0]).item():.4f} (deve ser estritamente 1.0)")


In [ ]:
# Criamos um lote de 4 imagens conceituais simples (RGB)
img_car = torch.clamp(torch.randn(1, 3, 224, 224) * 0.1 + torch.tensor([0.8, 0.2, 0.2]).view(1, 3, 1, 1), 0, 1).to(device)
img_cells = torch.clamp(torch.randn(1, 3, 224, 224) * 0.1 + torch.tensor([0.2, 0.4, 0.8]).view(1, 3, 1, 1), 0, 1).to(device)
img_nature = torch.clamp(torch.randn(1, 3, 224, 224) * 0.1 + torch.tensor([0.2, 0.8, 0.3]).view(1, 3, 1, 1), 0, 1).to(device)
batch_imgs = torch.cat([img_car, img_cells, img_nature], dim=0)

with torch.no_grad():
    image_features = clip_model.encode_image(batch_imgs)
    # Normalização L2 obrigatória
    image_features = F.normalize(image_features, p=2, dim=-1)

# Matriz de Similaridade de Cosseno: [Num_Imagens, Num_Queries]
similarity_matrix = (image_features @ text_features.T).cpu().numpy()

# Visualização da Matriz de Similaridade
plt.figure(figsize=(10, 4))
sns.heatmap(similarity_matrix, annot=True, fmt=".3f", cmap="Blues",
            xticklabels=[f"Q{i+1}: {q[:25]}..." for i, q in enumerate(queries)],
            yticklabels=["Img 1 (Vermelho/Carro)", "Img 2 (Azul/Células)", "Img 3 (Verde/Natureza)"])
plt.title("Matriz de Similaridade de Cosseno do CLIP (cos(θ))", fontsize=12, fontweight='bold')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()

# Calibração de Thresholding
THRESHOLD = 0.22
print(f"🎯 Aplicando Threshold de Decisão tau = {THRESHOLD}:")
for i in range(similarity_matrix.shape[0]):
    for j in range(similarity_matrix.shape[1]):
        score = similarity_matrix[i, j]
        if score >= THRESHOLD:
            print(f"   ✓ Detecção Positiva: Imagem {i+1} corresponde à Consulta {j+1} (Score: {score:.3f} >= {THRESHOLD})")


---
## ⚖️ Módulo 4: Governança em Desbalanceamento Severo, Recall e Avaliação Downstream

### 4.1 O Paradoxo da Acurácia Global
Demonstramos matematicamente como um classificador degenerado atinge 90% de acurácia global com **Recall = 0%** na classe rara.


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, recall_score, precision_score, f1_score

# Simulação de conjunto de teste com 1.000 amostras desbalanceadas:
# 900 Normais (Classe 0) e 100 Anomalias/Raras (Classe 1)
y_true = np.array([0] * 900 + [1] * 100)

# Modelo 1: Classificador Trivial Degenerado (Prediz sempre classe majoritária 0)
y_pred_trivial = np.zeros(1000, dtype=int)

acc_global = np.mean(y_true == y_pred_trivial)
recall_minority = recall_score(y_true, y_pred_trivial, pos_label=1)
f1_macro = f1_score(y_true, y_pred_trivial, average='macro')

print("🚨 Diagnóstico do Classificador Trivial sob Desbalanceamento:")
print(f"   • Acurácia Global:                {acc_global * 100:.1f}% (Parece excelente para leigos)")
print(f"   • Recall da Classe Crítica Rara:   {recall_minority * 100:.1f}% (Colapso total - 100 Falsos Negativos!)")
print(f"   • F1-Score Macro:                 {f1_macro * 100:.1f}%")

# Matriz de Confusão
cm = confusion_matrix(y_true, y_pred_trivial)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', cbar=False,
            xticklabels=['Pred: Normal (0)', 'Pred: Rara (1)'],
            yticklabels=['Real: Normal (0)', 'Real: Rara (1)'])
plt.title("Matriz de Confusão: Paradoxo da Acurácia", fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()


### 4.2 Protocolo de Validação Downstream com Aumento Generativo (cGAN/CycleGAN)

Demonstramos o protocolo correto: treinamos o classificador com e sem as amostras sintéticas da classe rara geradas por GAN, e avaliamos estritamente no conjunto de teste 100% real.


In [ ]:
# Dados sintéticos de treino: simulação de features extraídas
# Treino Base desbalanceado: 1.800 da classe 0, apenas 100 da classe 1
X_train_base = np.vstack([
    np.random.normal(loc=0.0, scale=1.0, size=(1800, 32)),
    np.random.normal(loc=1.2, scale=1.0, size=(100, 32))
])
y_train_base = np.array([0] * 1800 + [1] * 100)

# Conjunto de Teste 100% Real (Desbalanceado: 450 Classe 0, 50 Classe 1)
X_test_real = np.vstack([
    np.random.normal(loc=0.0, scale=1.0, size=(450, 32)),
    np.random.normal(loc=1.2, scale=1.0, size=(50, 32))
])
y_test_real = np.array([0] * 450 + [1] * 50)

# Modelo Linear Baseline treinado sem GAN
from sklearn.linear_model import LogisticRegression
clf_base = LogisticRegression()
clf_base.fit(X_train_base, y_train_base)
y_pred_base = clf_base.predict(X_test_real)
recall_base = recall_score(y_test_real, y_pred_base, pos_label=1)

# Treino Aumentado com GAN: adicionamos 800 amostras sintéticas da classe rara 1 geradas por cGAN
X_gen_cgan = np.random.normal(loc=1.2, scale=1.0, size=(800, 32))
y_gen_cgan = np.array([1] * 800)

X_train_augmented = np.vstack([X_train_base, X_gen_cgan])
y_train_augmented = np.concatenate([y_train_base, y_gen_cgan])

# Modelo treinado com dados aumentados generativamente
clf_augmented = LogisticRegression()
clf_augmented.fit(X_train_augmented, y_train_augmented)
# Avaliação OBRIGATORIAMENTE no mesmo conjunto de teste 100% REAL!
y_pred_aug = clf_augmented.predict(X_test_real)
recall_aug = recall_score(y_test_real, y_pred_aug, pos_label=1)

delta_recall = (recall_aug - recall_base) * 100
print(f"📈 Protocolo Downstream Concluído com Sucesso:")
print(f"   • Recall Base (sem GAN):       {recall_base * 100:.1f}%")
print(f"   • Recall Aumentado (com GAN):   {recall_aug * 100:.1f}%")
print(f"   • Salto no Recall (Δ Recall):   +{delta_recall:.1f}% pontos percentuais no teste REAL!")


---
## 🏆 Desafios Práticos Consolidados (Para Casa)

1. **Auditoria de Rescaling no FID:** Compare o FID calculando as features com interpolação bilinear versus bicúbica ao redimensionar para 299x299 e quantifique o impacto numérico.
2. **Attention Rollout no ViT:** Implemente a multiplicação em cadeia de matrizes de atenção através de todas as 12 camadas ($A_1 \times A_2 \times \dots \times A_{12}$) para visualizar o fluxo hierárquico de informação do patch inicial ao classificador final.
3. **Zero-Shot Prompt Ensembling:** No CLIP, calcule a média ponderada dos embeddings textuais de múltiplos templates para a mesma classe (ex: `["a photo of a {c}", "a close-up of a {c}", "a rendering of a {c}"]`) e meça a estabilização da similaridade de cosseno.
4. **GroupKFold vs Random Split:** Em um dataset com múltiplas instâncias do mesmo sujeito/sensor, compare a acurácia de validação obtida com divisão aleatória versus particionamento estrito por grupo e documente a presença de vazamento de dados.
